In [ ]:
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter

# ── Config — must match training exactly ─────────────────────
EMBEDDING_DIM   = 128
HIDDEN_DIM      = 128
NUM_LAYERS      = 1
DROPOUT         = 0.5
MAX_SRC_LEN     = 81
MAX_TGT_LEN     = 12
BEAM_WIDTH      = 5
BEAM_MAX_STEPS  = 12
CHECKPOINT_PATH = "checkpoints/best_model_v2"   
device          = torch.device('cpu')


# ── Vocabulary ────────────────────────────────────────────────
class Vocabulary:
    PAD_TOKEN = "<PAD>"; SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"; UNK_TOKEN = "<UNK>"
    PAD_IDX = 0; SOS_IDX = 1; EOS_IDX = 2; UNK_IDX = 3

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {}; self.idx2word = {}
        self.word_freq = Counter(); self.vocab_size = 0

    @staticmethod
    def tokenize(text):
        if not isinstance(text, str): return []
        return text.lower().strip().split()

    def encode(self, sentence, add_sos=False, add_eos=True):
        tokens = self.tokenize(sentence)
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in tokens]
        if add_sos: ids = [self.SOS_IDX] + ids
        if add_eos: ids = ids + [self.EOS_IDX]
        return ids

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f: return pickle.load(f)


# ── Model Classes ─────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, dropout, pad_idx=0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding  = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout    = nn.Dropout(dropout)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers,
                          dropout=dropout if num_layers > 1 else 0.0,
                          bidirectional=True, batch_first=True)
        self.fc  = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src, src_lens):
        embedded   = self.dropout(self.embedding(src))
        packed     = pack_padded_sequence(embedded, src_lens.cpu(),
                                          batch_first=True, enforce_sorted=True)
        packed_out, hidden = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_out, batch_first=True,
                                          total_length=MAX_SRC_LEN)
        outputs = (outputs[:, :, :self.hidden_dim] +
                   outputs[:, :, self.hidden_dim:])
        hidden  = torch.cat([
            torch.tanh(self.fc(torch.cat(
                [hidden[2*i], hidden[2*i+1]], dim=1
            ))).unsqueeze(0)
            for i in range(self.num_layers)
        ], dim=0)
        return outputs, hidden


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn_fc = nn.Linear(hidden_dim * 2, hidden_dim, bias=True)
        self.v       = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_lens):
        batch, src_len, _ = encoder_outputs.shape
        dec_h    = decoder_hidden.unsqueeze(1).expand(-1, src_len, -1)
        combined = torch.cat([dec_h, encoder_outputs], dim=2)
        energy   = self.v(torch.tanh(self.attn_fc(combined))).squeeze(2)
        mask     = torch.arange(src_len, device=src_lens.device)\
                       .unsqueeze(0) >= src_lens.unsqueeze(1)
        energy   = energy.masked_fill(mask, -1e10)
        attn_weights   = F.softmax(energy, dim=1)
        context_vector = torch.bmm(attn_weights.unsqueeze(1),
                                    encoder_outputs).squeeze(1)
        return attn_weights, context_vector


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, dropout, attention, pad_idx=0):
        super().__init__()
        self.attention  = attention
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.vocab_size = vocab_size
        self.embedding  = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout    = nn.Dropout(dropout)
        self.rnn = nn.GRU(embedding_dim + hidden_dim, hidden_dim, num_layers,
                          dropout=dropout if num_layers > 1 else 0.0,
                          bidirectional=False, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def tie_embeddings(self, encoder_embedding):
        self.embedding.weight = encoder_embedding.weight

    def forward(self, tgt_token, decoder_hidden, encoder_outputs, src_lens):
        embedded               = self.dropout(self.embedding(tgt_token.unsqueeze(1)))
        top_hidden             = decoder_hidden[-1]
        attn_weights, ctx      = self.attention(top_hidden, encoder_outputs, src_lens)
        rnn_input              = torch.cat([embedded, ctx.unsqueeze(1)], dim=2)
        output, decoder_hidden = self.rnn(rnn_input, decoder_hidden)
        prediction             = self.fc_out(output.squeeze(1))
        return prediction, decoder_hidden, attn_weights


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx=0, teacher_forcing_ratio=0.0):
        super().__init__()
        self.encoder               = encoder
        self.decoder               = decoder
        self.src_pad_idx           = src_pad_idx
        self.teacher_forcing_ratio = teacher_forcing_ratio

    def forward(self, src, tgt, src_lens):
        import random
        batch_size = src.shape[0]
        tgt_len    = tgt.shape[1]
        n_steps    = tgt_len - 1
        encoder_outputs, encoder_hidden = self.encoder(src, src_lens)
        decoder_hidden = encoder_hidden
        decoder_input  = tgt[:, 0]
        predictions = torch.zeros(n_steps, batch_size,
                                   self.decoder.vocab_size).to(src.device)
        attentions  = torch.zeros(n_steps, batch_size,
                                   src.shape[1]).to(src.device)
        for t in range(n_steps):
            prediction, decoder_hidden, attn_w = self.decoder(
                decoder_input, decoder_hidden, encoder_outputs, src_lens)
            predictions[t] = prediction
            attentions[t]  = attn_w
            use_tf = self.training and random.random() < self.teacher_forcing_ratio
            decoder_input = tgt[:, t+1] if use_tf else prediction.argmax(dim=1)
        return predictions, attentions


# ── Beam Search ───────────────────────────────────────────────
def beam_search_decode(model, src_sentence, vocab,
                       beam_width=BEAM_WIDTH, max_steps=BEAM_MAX_STEPS):
    model.eval()
    src_ids   = vocab.encode(src_sentence, add_sos=False, add_eos=True)
    src_len   = min(len(src_ids), MAX_SRC_LEN)
    src_ids   = src_ids[:MAX_SRC_LEN]
    src_ids  += [vocab.PAD_IDX] * (MAX_SRC_LEN - len(src_ids))
    src       = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0)
    src_len_t = torch.tensor([src_len], dtype=torch.long)

    with torch.no_grad():
        encoder_outputs, encoder_hidden = model.encoder(src, src_len_t)

    beams     = [(0.0, [vocab.SOS_IDX], encoder_hidden)]
    completed = []

    with torch.no_grad():
        for step in range(max_steps):
            new_beams = []
            for score, tokens, hidden in beams:
                last_token = torch.tensor([tokens[-1]], dtype=torch.long)
                prediction, new_hidden, _ = model.decoder(
                    last_token, hidden, encoder_outputs, src_len_t)
                log_probs = F.log_softmax(prediction.squeeze(0), dim=0)
                for prev_token in set(tokens):
                    if prev_token not in {vocab.SOS_IDX, vocab.PAD_IDX}:
                        log_probs[prev_token] -= 1.5
                top_log_probs, top_indices = log_probs.topk(beam_width)
                for log_p, idx in zip(top_log_probs.tolist(), top_indices.tolist()):
                    new_score  = score + log_p
                    new_tokens = tokens + [idx]
                    if idx == vocab.EOS_IDX:
                        length_penalty = len(new_tokens) ** 0.9
                        completed.append((new_score / length_penalty,
                                          new_tokens, new_hidden))
                    else:
                        new_beams.append((new_score, new_tokens, new_hidden))
            if not new_beams: break
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_width]

    if completed:
        completed.sort(key=lambda x: x[0], reverse=True)
        _, best_tokens, _ = completed[0]
    else:
        beams = [(s / (len(t) ** 0.9), t, h) for s, t, h in beams]
        beams.sort(key=lambda x: x[0], reverse=True)
        _, best_tokens, _ = beams[0]

    return " ".join([
        vocab.idx2word.get(t, vocab.UNK_TOKEN)
        for t in best_tokens
        if t not in {vocab.SOS_IDX, vocab.EOS_IDX, vocab.PAD_IDX}
    ])


# ── Load everything ───────────────────────────────────────────
def load_model():
    print("Loading vocabulary ...", end=" ", flush=True)
    vocab = Vocabulary.load("vocab/vocabulary.pkl")
    print(f"done  ({vocab.vocab_size:,} words)")

    print("Loading model ...", end=" ", flush=True)
    attention = Attention(hidden_dim=HIDDEN_DIM)
    encoder   = Encoder(vocab.vocab_size, EMBEDDING_DIM, HIDDEN_DIM,
                        NUM_LAYERS, DROPOUT, vocab.PAD_IDX)
    decoder   = Decoder(vocab.vocab_size, EMBEDDING_DIM, HIDDEN_DIM,
                        NUM_LAYERS, DROPOUT, attention, vocab.PAD_IDX)
    decoder.tie_embeddings(encoder.embedding)
    model     = Seq2Seq(encoder, decoder, vocab.PAD_IDX)

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device,
                             weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"done  (epoch {checkpoint['epoch']}, "
          f"val loss {checkpoint['val_loss']:.4f})")
    return model, vocab


# ── Main demo loop ────────────────────────────────────────────
def main():
    print()
    print("=" * 62)
    print("   Amazon Review Title Generator")
    print("   Seq2Seq + Bidirectional GRU + Bahdanau Attention")
    print("   Trained on 28,830 Amazon reviews  |  ~4M parameters")
    print("=" * 62)
    print()

    model, vocab = load_model()

    # ── Pre-loaded examples for team lead demo ────────────────
    examples = [
        ("Book — positive",
         "This book is absolutely fantastic. I have read it three times "
         "already and learn something new every time. The writing is clear "
         "and the examples are practical. Highly recommended to anyone "
         "interested in the subject. One of the best books I have ever read."),

        ("Book — negative",
         "Very disappointed with this book. The content is outdated and "
         "the author repeats the same points over and over. I expected "
         "much more based on the reviews. The chapters are poorly organised "
         "and it was hard to follow the logic. Would not recommend."),

        ("Electronics — positive",
         "These headphones are incredible. The sound quality is crystal "
         "clear and the bass is deep without being overwhelming. Battery "
         "lasts all day easily. Very comfortable even after hours of use. "
         "Best purchase I have made this year without question."),

        ("Electronics — negative",
         "Completely useless product. Stopped working after just three days. "
         "The build quality is terrible and feels very cheap. Customer "
         "service was no help at all. Total waste of money. Do not buy this."),

        ("Food — positive",
         "This coffee is outstanding. Rich deep flavour with no bitterness "
         "at all. I drink two cups every morning and it makes my whole day "
         "better. The aroma when you open the bag is incredible. Will "
         "definitely be ordering again very soon."),

        ("Movie / DVD — negative",
         "What a terrible film. The plot makes no sense and the acting is "
         "wooden throughout. I cannot believe this got such good reviews. "
         "The ending is completely unsatisfying. I want my two hours back. "
         "Save your money and watch something else."),
    ]

    print("─" * 62)
    print("  PRE-LOADED DEMO EXAMPLES")
    print("─" * 62)
    print()

    for i, (label, review) in enumerate(examples, 1):
        title = beam_search_decode(model, review, vocab)
        # Clean up UNK for presentation
        title_clean = title.replace("<UNK>", "...").strip()
        if not title_clean:
            title_clean = "(no output)"
        print(f"  [{i}] {label}")
        print(f"      Review : {review[:70]}...")
        print(f"      Title  : {title_clean}")
        print()

    # ── Interactive mode ──────────────────────────────────────
    print("─" * 62)
    print("  TYPE YOUR OWN REVIEW  (press Enter twice to generate)")
    print("  Type  'quit'  to exit")
    print("─" * 62)
    print()

    while True:
        print("  Paste your review below:")
        lines = []
        while True:
            line = input("  > ")
            if line.lower() == "quit":
                print()
                print("  Thank you for the demo!")
                print()
                return
            if line == "" and lines:
                break
            if line:
                lines.append(line)

        review = " ".join(lines)
        if not review.strip():
            continue

        print()
        print("  Generating title ...", end=" ", flush=True)
        title = beam_search_decode(model, review, vocab)
        title_clean = title.replace("<UNK>", "...").strip()
        if not title_clean:
            title_clean = "(model could not generate a title)"

        print("done")
        print()
        print(f"  ┌─────────────────────────────────────────┐")
        print(f"  │  Generated Title : {title_clean:<22}│")
        print(f"  └─────────────────────────────────────────┘")
        print()


if __name__ == "__main__":
    main()


   Amazon Review Title Generator
   Seq2Seq + Bidirectional GRU + Bahdanau Attention
   Trained on 28,830 Amazon reviews  |  ~4M parameters

Loading vocabulary ... done  (33,379 words)
Loading model ... done  (epoch 22, val loss 6.5778)
──────────────────────────────────────────────────────────────
  PRE-LOADED DEMO EXAMPLES
──────────────────────────────────────────────────────────────

  [1] Book — positive
      Review : This book is absolutely fantastic. I have read it three times already ...
      Title  : this book

  [2] Book — negative
      Review : Very disappointed with this book. The content is outdated and the auth...
      Title  : this book

  [3] Electronics — positive
      Review : These headphones are incredible. The sound quality is crystal clear an...
      Title  : great ...

  [4] Electronics — negative
      Review : Completely useless product. Stopped working after just three days. The...
      Title  : does not work

  [5] Food — positive
      Review : This 

  >  The product stopped working after one week. Complete waste of money.  Does not work as described. Very disappointed.
  >  



  Generating title ... done

  ┌─────────────────────────────────────────┐
  │  Generated Title : not ...               │
  └─────────────────────────────────────────┘

  Paste your review below:


  >  This book is a must read. One of the best books I have ever owned.  Buy this book immediately.
  >  



  Generating title ... done

  ┌─────────────────────────────────────────┐
  │  Generated Title : this book             │
  └─────────────────────────────────────────┘

  Paste your review below:


  >  This cd is perfect. Every song is amazing. Best album in years. Great music and great quality.
  >  



  Generating title ... done

  ┌─────────────────────────────────────────┐
  │  Generated Title : this cd               │
  └─────────────────────────────────────────┘

  Paste your review below:


  >  Do not buy this dvd. The picture quality is terrible and it skips constantly. Total waste of money.
  >  



  Generating title ... done

  ┌─────────────────────────────────────────┐
  │  Generated Title : this is               │
  └─────────────────────────────────────────┘

  Paste your review below:
